# 데이터셋 전처리

`260526_create_dataset`에서 생성한 xlsx를 임상적으로 유도 가능한 값으로 보완합니다.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from core.preprocessing_pipeline import PreprocessingConfig, PreprocessingPipeline

INPUT_DIR = Path("../outputs/260526_create_dataset")
OUTPUT_DIR = Path("../outputs/260526_preprocessing")
MISSING_RATE_THRESHOLD = 0.50  # 이상 결측 비율 (0.20 = 20%)
MIN_NUMERIC_RATIO = 0.85  # 비결측 중 이 비율 이상 수치 해석 가능 시 텍스트 공백

CONFIG = PreprocessingConfig(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    missing_rate_threshold=MISSING_RATE_THRESHOLD,
    min_numeric_ratio=MIN_NUMERIC_RATIO,
)


## 1. 전처리

| 대상 | 규칙 |
|------|------|
| (수치형 인자) | 비결측 값의 **90% 이상**이 수치로 해석되면, 나머지 텍스트에 대해 아래 순서 적용 후 **numeric** 변환 (요검사·메타 제외) |
| | ① `숫자 + 단위/괄호` 패턴 (`174.1 Cm`, `59.8 Kg`, `23.21(과체중)` 등): **수치 부분만 추출**하여 보존, 보고서에 `원본 → 수치 (N건)` 형식으로 출력 |
| | ② 추출 불가능한 순수 텍스트: **공백** 처리 |
| `혈압(수축기)`, `혈압(이완기)` | `이완기` ≥ `수축기`이면 두 값 교환 (컬럼 반대 기입) |
| `WBC` | K/µL. 값 ≥ 1000이면 /µL 오입력으로 보고 **÷1000**, 소수 둘째 자리 |
| `RBC` | M/µL. 값 ≥ 100이면 /µL 오입력으로 보고 **÷100**, 소수 둘째 자리 |
| `MCH`, `MCV` | `RBC`·`Hgb`·`Hct`·`MCV`·`MCH`·`MCHC` **6개 모두 유효**할 때, 아래 **뒤바뀜** 패턴이면 두 값 **교환**  |

**MCH·MCV 뒤바뀜 감지** (`RBC` ≠ 0, 허용 오차 ±0.5)

| 구분 | 조건 |
|------|------|
| 정상 | `\|MCH - Hgb/RBC×10\| ≤ 0.5` **그리고** `\|MCV - Hct/RBC×10\| ≤ 0.5` |
| 뒤바뀜 | 정상이 아니면서 `\|MCV - Hgb/RBC×10\| ≤ 0.5` **그리고** `\|MCH - Hct/RBC×10\| ≤ 0.5` → `MCH`↔`MCV` 교환 |


In [2]:
pipeline = PreprocessingPipeline(CONFIG)
pipeline.run_early_corrections()



[pre_diabetes] rows=1,240, cols=339
— 수치형 텍스트 공백 처리 (비결측 중 ≥85% 수치 해석 가능)


,field,numeric_ratio,extracted_count,blanked_count,dtype_before,dtype_after,extracted_values,excluded_values
0,AFP,0.9508,0,47,object,float64,,"음성 (36), Negative (4), 음성(negative) (3), < 2.00 (2), (1), <2.00 (1)"
1,CA19-9,0.9818,0,13,object,float64,,"< 2.0 (5), <2.00 (3), (2), < 0.01 (1), <2.7 (1), < 2.06 (1)"
2,CEA,0.9856,0,11,object,float64,,"음성 (5), < 2.4 (2), (1), 음성(negative) (1), <1.73 (1), < 0.01 (1)"
3,FSH,0.8571,0,1,object,float64,,(1)
4,N.RBC,0.9853,0,2,object,float64,,"0-3 (1), 0-5 (1)"
5,기타 골밀도 Tscore,0.9130,0,4,object,float64,,"정상 (2), 정상 소견입니다.\nRegion Z-score\nL1-L4 2.0 (1), 정상 소견입니다.\nRegion Z-score\nL1-L4 -1.7 (1)"
6,나이,0.9828,0,21,object,float64,,"M (14), F (7)"
7,비만도,0.9563,1,33,object,float64,23.21(과체중) → 23.21 (1건),"정상체중 (18), 비만1단계 (7), 과체중 (3), 표준 (2), 비만 (2), 경도비만 (1)"
8,신장,0.9992,1,0,object,float64,174.1 Cm → 174.1 (1건),
9,체중,0.9992,1,0,object,float64,59.8 Kg → 59.8 (1건),


— 혈압 반대 기입 교환 (수축기/이완기)


,field,swapped_count,swap_detail
0,혈압(수축기)·혈압(이완기),4,"61/105 → 105/61 (1), 75/115 → 115/75 (1), 68/110 → 110/68 (1), 86/115 → 115/86 (1)"


— WBC 단위 보정 (≥1000 → ÷1000, K/µL)


...

,field,fixed_count,fix_detail
0,WBC,7,"5120 → 5.12 (1), 4630 → 4.63 (1), 9010 → 9.01 (1), 7200 → 7.20 (1), 6320 → 6.32 (1), 7780 → 7.78 (1), 8380 → 8.38 (1)"


— RBC 단위 보정 (≥100 → ÷100, M/µL)


,field,fixed_count,fix_detail
0,RBC,7,"443 → 4.43 (1), 485 → 4.85 (1), 506 → 5.06 (1), 574 → 5.74 (1), 502 → 5.02 (1), 581 → 5.81 (1), 459 → 4.59 (1)"


— MCH·MCV 뒤바뀜 교환


,field,swapped_count,swap_detail
0,MCH·MCV,2,"MCH 91.3→32.1 / MCV 32.1→91.3 (1), MCH 90.2→30.2 / MCV 30.2→90.2 (1)"


| 대상 | 변환 |
|------|------|
| `Bilirubin`, `Blood`, `Glucose`, `Keton`, `Leukocyte`, `Nitrite`, `Protein`, `Urobilinogen`, `HBs-Ab`, `HBs-Ag` | `음성` / `양성` (분류 불가 → 결측) |


In [3]:
pipeline.run_lab_encoding()



— Bilirubin
  양성: 양성(1+) (9), 1 Positive (4), 약양성(+/-) (2), 양성(+) (2), +- (1)
  음성: 음성 (970), Negative (88), - (50), 음 성 (16), 음성(-) (8), 음성(negative) (7), neg (3), Negative  (1)
  NaN: NaN (78), 1.00 (1)

— Blood
  양성: 약양성(+/-) (32), 양성(+) (18), 양성(3+) (15), 양성(2+) (12), 양성(1+) (10), 약양성 (9), 양성 (8), 양성(+++) (7), 양성(+1) (6), + (5), 2+ (4), Trace (4), 1 Positive (3), 약양성(+-) (3), 양성(+2) (3), 양성(++) (3), 3 양성 (2), 양성(++++) (2), 1+ (2), +- (2), 양성(4+) (2), 1Positive (1), 2 Positive (1), Trace(+-) (1), 2Positive (1), 3 Positive (1), 3+ (1), 양성(positive)(++++) (1)
  음성: 음성 (834), Negative (78), - (43), 음 성 (22), 음성(-) (8), 음성(negative) (6), Negative  (1), neg (1)
  NaN: NaN (86), 4 (1), 3 (1)

— Glucose
  양성: +- (1)
  음성: 음성 (1019), Negative (92), - (40), 음 성 (17), 음성(-) (8), 음성(negative) (7), neg (6), Normal (3), Negative  (1)
  NaN: NaN (46)

— Keton
  양성: 양성(1+) (32), 양성(2+) (31), 약양성(+/-) (30), 양성 (17), 약양성 (13), 양성(3+) (12), 양성(++) (11), 1 Positive (8), 양성(+) (6), 2 Positive (6), 약양성

### 임상 불가능 수치 공백 처리 (생리학적·최대 보수)

정성 검사 인코딩 **직후**, 임상·결측 보완 **이전**·**§2 고결측 제외 이전**에 **수치형 인자**에 대해 **생리학적으로 불가능하거나 명백한 입력 오류**만 결측(공백)으로 둡니다.

- **최대 보수 원칙**: 생리학적으로 나올 수 있는 범위를 넓게 두고, 경계성·고위험 비정상 수치는 **유지**
- **제거 대상**: 센티널(`999` 계열), 혈압 논리 오류(전처리 교환 후 잔여), 생리학적 불가능 극단값 (`WBC` ≥1000·`RBC` ≥100 /µL 오입력은 §1에서 보정)

### 공통 (항상 적용)

| 구분 | 기준 |
|------|------|
| 센티널 | `999`, `999.9`, `9999`, `99999` |
| 논리 오류 | `혈압(이완기)` ≥ `혈압(수축기)` → 두 혈압 모두 공백 |

### 인자별 허용 범위 (`ClinicalRangeBlanker.DEFAULT_RULES`, 최대 보수)

| 구분 | 인자 | 최소 | 최대 | 비고 |
|------|------|------|------|------|
| 인구·체형 | `나이` | 0 | 150 | 세 |
|  | `신장` | 50 | 250 | cm |
|  | `체중` | 10 | 500 | kg |
|  | `BMI` | 5 | 60 | kg/m² |
|  | `비만도` | 0 | 500 | % |
|  | `허리둘레` | 10 | 500 | cm |
| 혈압 | `혈압(수축기)` | 50 | 250 | mmHg |
|  | `혈압(이완기)` | 30 | 150 | mmHg |
| 당뇨 | `공복혈당` | 10 | 1000 | mg/dL |
| 지질 | `T.Cholesterol` | 30 | 1000 | — |
|  | `HDL` | 1 | 300 | — |
|  | `LDL` | 1 | 600 | — |
|  | `Triglyceride` | 1 | 10000 | — |
| 간 | `GOT(AST)`, `GPT(ALT)` | 0 | 10000 | U/L |
|  | `r-GTP` | 0 | 5000 | — |
|  | `ALP` | 0 | 2000 | — |
|  | `T.Bilirubin` | 0 | 50 | — |
|  | `D.Bilirubin` | 0 | 30 | — |
| 신장 | `Creatinine` | 0.1 | 30 | — |
|  | `e-GFR` | 0 | 500 | mL/min/1.73m² |
|  | `BUN` | 0.5 | 200 | — |
|  | `Uric acid` | 0.1 | 30 | — |
| 혈액 | `WBC` | 0.1 | 200 | K/uL |
|  | `RBC` | 0.5 | 15 | M/uL |
|  | `Hgb` | 1 | 30 | — |
|  | `Hct` | 5 | 90 | — |
|  | `Platelet` | 1 | 5000 | 10³/μL |
|  | `MCV` | 30 | 150 | — |
|  | `MCH` | 10 | 100 | — |
|  | `MCHC` | 20 | 50 | — |
|  | `RDW` | 3 | 60 | — |
|  | `MPV` | 3 | 25 | — |
|  | `PDW` | 5 | 80 | — |
| 혈액 | `Lymphocyte`, `Monocyte`, `Eosinophil`, `Basophil` | 0 | 100 | % |
|  | `B/C ratio` | 0 | 100 | — |
| 소변 | `PH` | 3 | 14 | — |
|  | `SG` | 1.0 | 1.06 | — |
| 기타 | `TSH` | 0 | 500 | — |
|  | `T.Protein` | 2 | 15 | — |
|  | `Albumin` | 1 | 8 | — |
|  | `Globulin` | 0.5 | 10 | — |
|  | `A/G ratio` | 0.1 | 25 | — |

`음성`/`양성` 인코딩 인자, `gender`, `label`, `interval_days` 등은 대상에서 제외합니다.


In [4]:
pipeline.run_range_blanking() 


[pre_diabetes] 임상 불가능 수치 공백 처리


,field,below_min,above_max,sentinel,logical,total_blanked,excluded_values
0,BMI,1,3,0,0,4,"0 (1), 105 (1), 106 (1), 130 (1)"
1,체중,0,1,1,0,1,999.9 (1)
2,허리둘레,0,1,1,0,1,999.9 (1)
3,공복혈당,1,0,0,0,1,5.6 (1)
4,Hgb,1,0,0,0,1,0.8 (1)
5,Hct,1,0,0,0,1,3.2 (1)



[diabetes] 임상 불가능 수치 공백 처리


,field,below_min,above_max,sentinel,logical,total_blanked,excluded_values
0,LDL,1,0,0,0,1,0 (1)
1,MCV,1,0,0,0,1,9.2 (1)


### 임상·결측 보완

공백 처리 **이후**에 적용합니다. 입력 수치에 센티널(`999`, `999.9`, `9999`, `99999`)이 있으면 해당 행의 대체(BMI·지질·e-GFR 등)는 수행하지 않습니다.

결측치 처리

| 대상 컬럼 | 규칙 |
|-----------|------|
| `나이` | `나이` = (`checkup_date` - `birthday`).days / 365.25 |
| `gender` | `gender` 1·`성별` M/남/남자 → `남자`, `gender` 2·`성별` F/여/여자 → `여자`. 통합 후 `성별` 제거 |
| `BMI` | `BMI` = `체중` / (`신장` / 100)² |
| `T.Cholesterol` | `T.Cholesterol` = `LDL` + `HDL` + int(`Triglyceride`/5), `Triglyceride` < 400 |
| `HDL` | `HDL` = `T.Cholesterol` - `LDL` - int(`Triglyceride`/5), `Triglyceride` < 400 |
| `Triglyceride` | `Triglyceride` = 5×(`T.Cholesterol`-`LDL`-`HDL`), 결과 < 400 |
| `LDL` | `LDL` = `T.Cholesterol` - `HDL` - int(`Triglyceride`/5), `Triglyceride` < 400 |
| `Globulin` | `Globulin` = `T.Protein` - `Albumin` |
| `Albumin` | `Albumin` = `T.Protein` - `Globulin` |
| `T.Protein` | `T.Protein` = `Albumin` + `Globulin` |
| `A/G ratio` | `A/G ratio` = `Albumin` / `Globulin` |
| `UIBC` | `UIBC` = `TIBC` - `Fe` |
| `철포화율` | `철포화율` = `Fe` / `TIBC` × 100 |
| `비만도` | `비만도` = (`체중` / 표준체중) × 100, 표준체중 = (`신장` - 100) × (0.9 if `gender`=`남자` else 0.85) |
| `e-GFR` | CKD-EPI(2009) 공식으로 계산 (아래 참고) |
| `MCH` | `MCH` = `Hgb` / `RBC` × 10 (`RBC` ≠ 0) |
| `MCHC` | `MCHC` = `Hgb` / `Hct` × 100 (`Hct` ≠ 0) |
| `MCV` | `MCV` = `Hct` / `RBC` × 10 (`RBC` ≠ 0) |
| `Hct` | `Hct` = `RBC` × `MCV` / 10 |


### `e-GFR` — CKD-EPI(2009)


**대상·입력 컬럼**

| 구분 | 컬럼 |
|------|------|
| 채우는 컬럼 | `e-GFR` (이미 값이 있으면 덮어쓰지 않음) |
| 입력 | `Creatinine`, `나이`, `gender` (`남자` / `여자`) |


**기호 (`Creatinine` 단위: mg/dL)**

| 기호 | 의미 |
|------|------|
| $Cr$ | `Creatinine` |
| $Age$ | `나이` |
| $\kappa,\ \alpha,\ s$ | `gender`에 따른 상수 (아래 표) |

**성별별 상수**

| | `gender` = `여자` | `gender` = `남자` |
|--|--|--|
| $\kappa$ | 0.7 | 0.9 |
| $\alpha$ | −0.329 | −0.411 |
| $s$ (성별계수) | 1.018 | 1.0 |

**공식 (`core/preprocessor.py` → `_ckd_epi_2009`와 동일)**

$Cr \le \kappa$ 일 때:

$$eGFR = 141 \times \min(Cr/\kappa, 1)^{\alpha} \times \max(Cr/\kappa, 1)^{-0.329} \times 0.993^{Age} \times s$$

$Cr > \kappa$ 일 때:

$$eGFR = 141 \times \min(Cr/\kappa, 1)^{-1.209} \times \max(Cr/\kappa, 1)^{-1.209} \times 0.993^{Age} \times s$$


In [5]:
pipeline.run_imputations()


[pre_diabetes] 임상·결측 보완
— 결측 비율 (공백 처리 후 → 임상 보완 후)


,field,rows,missing_before,missing_pct_before,missing_after,missing_pct_after,filled,pct_point_change
0,나이,1240,41,3.31%,0,0.00%,41,-3.31%p
1,BMI,1240,176,14.19%,7,0.56%,169,-13.63%p
2,T.Cholesterol,1240,15,1.21%,15,1.21%,0,+0.00%p
3,HDL,1240,37,2.98%,32,2.58%,5,-0.40%p
4,Triglyceride,1240,19,1.53%,15,1.21%,4,-0.32%p
5,LDL,1240,41,3.31%,33,2.66%,8,-0.65%p
6,Globulin,1240,113,9.11%,58,4.68%,55,-4.43%p
7,Albumin,1240,53,4.27%,52,4.19%,1,-0.08%p
8,T.Protein,1240,63,5.08%,58,4.68%,5,-0.40%p
9,A/G ratio,1240,128,10.32%,58,4.68%,70,-5.64%p



[diabetes] 임상·결측 보완
— 결측 비율 (공백 처리 후 → 임상 보완 후)


,field,rows,missing_before,missing_pct_before,missing_after,missing_pct_after,filled,pct_point_change
0,나이,629,10,1.59%,0,0.00%,10,-1.59%p
1,BMI,629,115,18.28%,1,0.16%,114,-18.12%p
2,T.Cholesterol,629,12,1.91%,12,1.91%,0,+0.00%p
3,HDL,629,15,2.38%,15,2.38%,0,+0.00%p
4,Triglyceride,629,12,1.91%,12,1.91%,0,+0.00%p
5,LDL,629,24,3.82%,18,2.86%,6,-0.96%p
6,Globulin,629,73,11.61%,30,4.77%,43,-6.84%p
7,Albumin,629,28,4.45%,28,4.45%,0,+0.00%p
8,T.Protein,629,37,5.88%,31,4.93%,6,-0.95%p
9,A/G ratio,629,75,11.92%,30,4.77%,45,-7.15%p


In [6]:
pipeline.run_range_blanking() 


[pre_diabetes] 임상 불가능 수치 공백 처리
  공백 처리된 인자 없음

[diabetes] 임상 불가능 수치 공백 처리
  공백 처리된 인자 없음


## 2. 고결측 컬럼 제외

아래 코드 셀의 **설정**에서 결측 임계값·유지 컬럼을 조정합니다. 메타 컬럼(`user_key`, `label`, `selected_transition`, `full_transition` 등)과 `HBs-Ab`, `HBs-Ag`는 결측률과 관계없이 유지됩니다.


In [7]:
# ===== 고결측 컬럼 제외 설정 =====
EXTRA_PRESERVE_COLUMNS: list[str] = ["당화혈색소", "HbA1C"]  # 고결측 제외 대상에서 제외(항상 유지)
EXTRA_ALWAYS_DROP_COLUMNS: list[str] = [
    "흉부 X-RAY(정면)",
    "심전도",
    "청력우",
    "청력좌",
    "birthday",
    "백혈구(소변현미경)",
    "적혈구(소변현미경)",
    "biz_type",
    "갑상선 초음파",
    "상복부 초음파",
    "biz_type",
    "HCV Ab",
    "R-A인자",
    "안저"
]  # 결측률과 무관하게 제외

pipeline.configure_column_filter(
    extra_preserve_columns=frozenset(EXTRA_PRESERVE_COLUMNS),
    extra_always_drop_columns=frozenset(EXTRA_ALWAYS_DROP_COLUMNS),
)
pipeline.run_column_filter()


항상 제외 컬럼: ['HCV Ab', 'R-A인자', 'birthday', 'biz_type', '갑상선 초음파', '백혈구(소변현미경)', '상복부 초음파', '심전도', '안저', '적혈구(소변현미경)', '청력우', '청력좌', '흉부 X-RAY(정면)']

[pre_diabetes] 결측 ≥ 50% 인자 제외
  컬럼 수: 338 → 91 (제거 247개)


,field,missing_count,missing_pct
0,Anti Thyroglobulin Ab,1239,99.92
1,M2PK,1239,99.92
2,Methemoglobin,1239,99.92
3,PB smear,1239,99.92
4,Pb,1239,99.92
...,...,...,...
242,백혈구(소변현미경),334,26.94
243,청력우,228,18.39
244,청력좌,228,18.39
245,심전도,109,8.79



[diabetes] 결측 ≥ 50% 인자 제외
  컬럼 수: 316 → 96 (제거 220개)


,field,missing_count,missing_pct
0,ABI,628,99.84
1,LH,628,99.84
2,M2PK,628,99.84
3,NT-PrpBNP,628,99.84
4,PEF(L/S),628,99.84
...,...,...,...
215,적혈구(소변현미경),132,20.99
216,청력우,126,20.03
217,청력좌,126,20.03
218,심전도,56,8.90


## 3. 저장


In [8]:
pipeline.run_export()



[pre_diabetes] 저장 전 DataFrame
  행=1,240, 열=91
  numeric (72개): label, interval_days, A/G ratio, AFP, ALP, Albumin, B/C ratio, BMI, BUN, Basophil, CA19-9, CEA, CPK, Creatinine, D.Bilirubin, Eosinophil, Fe, Free T4, GOT(AST), GPT(ALT), Globulin, HDL, Hct, Hgb, I.Bilirubin, LDH, LDL, Lymphocyte, MCH, MCHC, MCV, MPV, Monocyte, Neutroph, PCT, PDW, PH, Platelet, RBC, RDW, SG, T-Amylase, T.Bilirubin, T.Cholesterol, T.Protein, TIBC, TSH, Triglyceride, UIBC, Uric acid, WBC, current_glucose, e-GFR, future_glucose, glucose_change, r-GTP, 공복혈당, 나이, 당화혈색소, 맥박, 비만도, 시력(우안), 시력(좌안), 신장, 안압(우안), 안압(좌안), 청력우(1000Hz), 청력좌(1000Hz), 체중, 허리둘레, 혈압(수축기), 혈압(이완기)
  object (19개): user_key, current_checkup_date, future_checkup_date, selected_transition, full_transition, checkup_date, gender, birthday, Bilirubin, Blood, CRP, Glucose, HBs-Ab, HBs-Ag, Keton, Leukocyte, Nitrite, Protein, Urobilinogen
Saved: ../outputs/260526_preprocessing/pre_diabetes_dataset.xlsx  (1,240명, 91컬럼)

[diabetes] 저장 전 DataFrame
  행=629